# Neural Decoding of Decision-Making Behavior

This notebook analyzes neural population activity recorded during a decision-making task. Using spike data and behavioral trial information, we test whether neural firing rates can predict behavioral choice. The analysis includes data preprocessing, visualization of neural activity patterns, dimensionality reduction using Principal Component Analysis (PCA), and neural decoding using logistic regression. We also examine how decoding performance varies across anatomical brain regions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1. Data Wrangling

In this section we load neural spike data, trial information, and neuron metadata. The dataset includes spike times associated with neurons and trials, behavioral variables such as stimulus condition and choice, and anatomical brain region labels for each neuron.

The goal of preprocessing is to convert spike timing data into firing rates for each neuron during each trial. These firing rates are then organized into a population activity matrix where rows represent trials and columns represent neurons. This matrix serves as the input for downstream dimensionality reduction and decoding analyses.

In [ ]:
spikes = pd.read_csv("../data/spikes.csv")
trials = pd.read_csv("../data/trials.csv")
neurons = pd.read_csv("../data/neurons.csv")

print("Spikes shape:", spikes.shape)
print("Trials shape:", trials.shape)
print("Neurons shape:", neurons.shape)

In [ ]:
print(spikes.isnull().sum())
print(trials.isnull().sum())
print(neurons.isnull().sum())

The dataset contains observations across multiple trials and neurons. We verify that no missing values are present in the dataset before proceeding with analysis. Handling missing data is an important preprocessing step because incomplete observations can bias downstream analyses.

In [ ]:
population_matrix = spikes.groupby(["trial_id","neuron_id"]).size().unstack(fill_value=0)

population_matrix.head()

The population activity matrix represents neural firing rates for each neuron during each trial. Rows correspond to trials and columns correspond to neurons. Each value represents the number of spikes recorded for a given neuron during a given trial.

## 2. Integrating Neural Activity with Anatomical Regions

Neurons recorded in the dataset originate from different anatomical brain regions. By linking each neuron to its corresponding region, we can examine whether certain areas contribute more strongly to neural representations of decision-making.

In [ ]:
region_counts = neurons["brain_region"].value_counts()

region_counts.plot(kind="bar")
plt.xlabel("Brain Region")
plt.ylabel("Number of Neurons")
plt.title("Distribution of Neurons Across Brain Regions")
plt.show()

This plot shows the number of neurons recorded from each anatomical brain region. Differences in neuron counts across regions may influence how strongly each region contributes to decoding analyses.

## 3. Behavioral Data Visualization

In [ ]:
trials["stimulus"].value_counts().plot(kind="bar")

plt.xlabel("Stimulus Condition")
plt.ylabel("Number of Trials")
plt.title("Distribution of Stimulus Conditions")
plt.show()

In [ ]:
plt.hist(trials["response_time"], bins=15)

plt.xlabel("Response Time")
plt.ylabel("Count")
plt.title("Distribution of Response Times")
plt.show()

The histogram shows the distribution of behavioral response times across trials. Variability in response times reflects differences in decision processing across trials.

## 4. Dimensionality Reduction (PCA)

Neural population recordings contain activity from many neurons simultaneously, producing high-dimensional datasets. Principal Component Analysis (PCA) reduces this complexity by identifying directions of maximum variance in neural activity. Projecting neural data onto the first few principal components allows us to visualize patterns in neural population activity.

In [ ]:
X = population_matrix.values

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

In [ ]:
plt.scatter(X_pca[:,0], X_pca[:,1], c=trials["choice"])

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of Neural Population Activity")
plt.show()

Each point represents neural population activity during a single trial projected onto the first two principal components. Coloring points by behavioral choice allows us to examine whether neural activity patterns differ between decisions.

## 5. Neural Decoding with Logistic Regression

To test whether neural activity predicts behavioral choice, we train a logistic regression classifier. The classifier learns to associate patterns of neural firing with behavioral outcomes. Decoding accuracy measures how well neural activity predicts the animal's choice.

In [ ]:
y = trials["choice"]

X_train, X_test, y_train, y_test = train_test_split(
    population_matrix, y, test_size=0.3
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Decoding accuracy:", accuracy)

In [ ]:
plt.bar(["Decoder Accuracy"], [accuracy])

plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.title("Neural Decoding Performance")
plt.show()

Decoding accuracy above chance levels suggests that neural population activity contains information about behavioral decisions.

## 6. Permutation Test for Statistical Significance

To determine whether decoding accuracy is significantly greater than chance, we perform a permutation test. Behavioral labels are randomly shuffled across trials and decoding accuracy is recomputed multiple times. This generates a null distribution of decoding performance expected by chance.

In [ ]:
perm_acc = []

for i in range(200):

    shuffled = np.random.permutation(y)

    X_train, X_test, y_train, y_test = train_test_split(
        population_matrix, shuffled, test_size=0.3
    )

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    perm_acc.append(accuracy_score(y_test, pred))

In [ ]:
plt.hist(perm_acc, bins=20)
plt.axvline(accuracy)

plt.xlabel("Accuracy")
plt.ylabel("Count")
plt.title("Permutation Test Null Distribution")
plt.show()

The permutation test provides a baseline distribution of decoding accuracies expected if neural activity and behavior were unrelated. If the observed decoding accuracy lies outside this distribution, it indicates that neural activity contains meaningful information about behavioral choice.